# T20 Score Prediction — Base Model Evaluation

Evaluating the base LLM (`meta-llama/Llama-3.2-3B`) zero-shot against our T20 first-innings score prediction dataset.

The model receives the in-game state prompt and must predict the final innings score with no fine-tuning.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login

# Add research/ dir to path so util.py is importable
sys.path.insert(0, str(Path().resolve()))

from util import evaluate

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)
print("Logged in to Hugging Face")

In [ ]:
from datasets import load_dataset

HF_DATASET = "ratnayakatilanka/t20_score_prediction_prompts"

dataset = load_dataset(HF_DATASET)
print(dataset)

# Use the test split for evaluation
test_data = list(dataset["test"])
print(f"\nTest set size: {len(test_data):,} rows")
print("\n--- Sample datapoint ---")
print(test_data[0]["prompt"])
print("\n--- Completion ---", test_data[0]["completion"])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Llama-3.2-3B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.eval()
print(f"Loaded: {BASE_MODEL}")

In [ ]:
def llama_base(datapoint: dict) -> str:
    """Run the base LLaMA model on a single datapoint and return the generated text."""
    prompt = datapoint["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (strip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Quick sanity check
sample = test_data[0]
raw_output = llama_base(sample)
print(f"Raw output : '{raw_output}'")
print(f"Actual     : {sample['completion']}")

In [ ]:
evaluate(llama_base, test_data, size=min(200, len(test_data))) 

#Colab link:

https://colab.research.google.com/drive/1CIoBec_1CuRraEzCNXfPM0022nhUmHLu?usp=sharing